# Udeco 专用交互绘图

选择 `PPG_sensor_env` 内核，修改下方 `SAMPLE_CSV` 后从上至下运行。算法直接复用
`udeco_uh_roi_edge_detection.py`，不需要先执行原分析 notebook。

十类图分别运行；同类的两路独立图也各占一个单元格。滚轮缩放、拖拽平移、双击恢复视图；
合并图的时间轴同步。重新绘图不重复分析，修改检测参数后需重新运行计算单元格。

样式参考 [Nature figure guide](https://research-figure-guide.nature.com/figures/preparing-figures-our-specifications/)，
适用于 notebook 屏幕阅读。PPG/ACC 保留设备采集数值（未标定物理单位）；Udeco 为无量纲比值。

## 1. 手动选择样本与参数

In [ ]:
# 从仓库根目录或 figures/ 启动均可；只导入本工作树的模块。
from pathlib import Path
import os
import sys

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "udeco_uh_roi_edge_detection.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("请从仓库根目录或 figures/ 目录启动 Jupyter。")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pathlib import Path

# 在这里手动输入所选 *_sensor.csv 的路径（绝对路径推荐）。
SAMPLE_CSV = Path(os.environ.get("UDECO_DATA_ROOT", str(PROJECT_ROOT / "subject_data"))) / "cgx" / "cgx_rope_1_sensor.csv"
# None：从样本父目录向上查找最近的 ck.mat；也可填写包含 ck.mat 的目录。
DATA_ROOT = None
FS = 100
PLOT_START_SEC = None                 # None 表示记录起点
PLOT_END_SEC = None                   # None 表示记录终点
MOTION_START_SEC = None               # 手动指定时，起止必须同时填写
MOTION_END_SEC = None
F0_OVERRIDE_HZ = {2: None, 3: None}
BASELINE_SEC = 15.0
CONTEXT_WINDOW_SEC = 10.0
CONTEXT_ROBUST_ITERS = 1
FFT_VIEW_HZ = (0.0, 10.0)
# 其余检测参数保持 ExperimentConfig 默认值；可在这里按字段名覆盖。
DETECTOR_OVERRIDES = {}

## 2. 导入与统一样式

In [ ]:
import string
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display
from udeco_analysis import _repair_pydisplay_sequence
from udeco_edge_detection import CascadeStateConfig, estimate_context_trend
from udeco_uh_roi_edge_detection import (
    ExperimentConfig, load_paired_sample, preprocess_channel, baseline_sigmas,
    resolve_motion_interval, estimate_motion_frequency, find_uh_rois,
    detect_roi_candidates, localize_roi_candidates, recover_missing_extrema_pairs,
    build_direct_event_table,
)

pio.renderers.default = "plotly_mimetype"
COLORS = {"PPG_G": "#238B59", "PPG_R": "#C95040", "PPG_IR": "#8060A3",
          "Uh2": "#0072B2", "Uh3": "#D55E00", "Udeco2": "#0072B2",
          "Udeco3": "#D55E00", "ACC_X": "#298F85", "ACC_Y": "#BD7B2E",
          "ACC_Z": "#8060A3"}
STATE_COLORS = {"detach-attach": "#DAA083", "release-force": "#88B5CE", "slide": "#BCBEC3"}
EDGE_COLORS = {1: "#B8503E", -1: "#2769A0"}
PPG_NAMES = ["PPG_G", "PPG_R", "PPG_IR"]
ACC_NAMES = ["ACC_X", "ACC_Y", "ACC_Z"]
FIGURES = {}
SHOW_CONFIG = {"scrollZoom": True, "displaylogo": False, "responsive": True,
               "toImageButtonOptions": {"format": "png", "scale": 3}}

def show_figure(key, fig, *, static=False):
    FIGURES[key] = fig
    fig.show(config={"staticPlot": True, "displayModeBar": False} if static else SHOW_CONFIG)

## 3. 加载、时间轴修复与预处理

PPG/ACC 使用居中的 4 点滑动平均（边界采用可用点）；Uh/Udeco 使用原算法的 4 点中值滤波。
PPG 基线和 Uh/Udeco 基线调用同一个 LOWESS 方法。图 8 不计算或展示 ACC 基线。
缺失或无穷数值先转换为缺失值再线性插值；完全无有效数据的通道会明确报错。

In [ ]:
def resolve_input(sensor_csv, data_root=None):
    path = Path(sensor_csv).expanduser().resolve()
    if not path.is_file():
        raise FileNotFoundError(f"样本文件不存在：{path}")
    root = Path(data_root).expanduser().resolve() if data_root is not None else next(
        (parent for parent in path.parents if (parent / "ck.mat").is_file()), None)
    if root is None or not (root / "ck.mat").is_file():
        raise FileNotFoundError("未找到 ck.mat；请将 DATA_ROOT 设置为标定文件所在目录。")
    return path, root

def numeric_signal(frame, name):
    if name not in frame:
        raise KeyError(f"CSV 缺少必需通道：{name}")
    values = pd.to_numeric(frame[name], errors="coerce").replace([np.inf, -np.inf], np.nan)
    if not values.notna().any():
        raise ValueError(f"通道 {name} 没有有效数值")
    return values.interpolate(limit_direction="both").to_numpy(dtype=float)

def moving_average4(values):
    return pd.Series(values).rolling(4, center=True, min_periods=1).mean().to_numpy()

def time_limits(time, start=None, end=None):
    lo = float(time[0]) if start is None else float(start)
    hi = float(time[-1]) if end is None else float(end)
    if not np.isfinite([lo, hi]).all() or not time[0] <= lo < hi <= time[-1]:
        raise ValueError("显示范围必须满足：记录起点 <= 开始时间 < 结束时间 <= 记录终点。")
    return lo, hi

sensor_path, data_root = resolve_input(SAMPLE_CSV, DATA_ROOT)
if not isinstance(FS, (int, np.integer)) or FS <= 0:
    raise ValueError("FS 必须为正整数。")
cfg_kwargs = dict(median_kernel=4, baseline_sec=BASELINE_SEC,
                  context_window_sec=CONTEXT_WINDOW_SEC,
                  context_robust_iters=CONTEXT_ROBUST_ITERS)
cfg_kwargs.update(DETECTOR_OVERRIDES)
config = ExperimentConfig(**cfg_kwargs)
sample = load_paired_sample(sensor_path, data_root=data_root, fs=FS)
time_s, fs = sample["time_s"], sample["fs"]
repaired, _, _ = _repair_pydisplay_sequence(pd.read_csv(sensor_path))
if len(repaired) != len(time_s):
    raise ValueError("PPG/ACC 与 Uh/Udeco 修复后的时间轴长度不一致。")
time_limits(time_s, PLOT_START_SEC, PLOT_END_SEC)
processed = {name: preprocess_channel(values, fs, config)
             for name, values in sample["signals"].items()}
smoothed = {name: moving_average4(numeric_signal(repaired, name))
            for name in PPG_NAMES + ACC_NAMES}
baseline_config = CascadeStateConfig(context_window_s=config.context_window_sec,
                                    context_robust_iterations=config.context_robust_iters)
baselines = {name: values["Bcontext"] for name, values in processed.items()}
baselines.update({name: estimate_context_trend(smoothed[name], fs, baseline_config)["Bcontext"]
                  for name in PPG_NAMES})
print(f"{sample['sample_name']} | {len(time_s):,} samples | {fs} Hz | {time_s[-1]:.2f} s")
print(f"Calibration: {data_root / 'ck.mat'}")

## 4. 计算 ROI、边沿和状态（只运行一次）

每路独立完成主频、检峰、ROI、边沿定位、缺失边沿补全和分类。
图 6–8 的公共背景使用 Uh2 ROI，但不改变 Udeco3 的检测结果。
`slide` 是原 `cancelled` 的展示别名，表示原算法未接受为另外两种 ROI 模式；不是新增分类器。

In [ ]:
noise = {ch: baseline_sigmas(processed[f"Uh{ch}"]["edge_signal"],
                            processed[f"Udeco{ch}"]["edge_signal"], fs=fs,
                            baseline_sec=config.baseline_sec) for ch in (2, 3)}
motion = resolve_motion_interval(sample, baseline_sec=config.baseline_sec,
                                 manual_start_sec=MOTION_START_SEC, manual_end_sec=MOTION_END_SEC)
roi_results, detections, localized, tables = {}, {}, {}, {}
direction_map = {1: {"sharp": "detach", "gradual": "release"},
                 -1: {"sharp": "attach", "gradual": "force"}}
for ch in (2, 3):
    uh, ud = processed[f"Uh{ch}"], processed[f"Udeco{ch}"]
    spectrum = estimate_motion_frequency(uh["edge_signal"], fs=fs, motion_interval=motion,
                                         min_hz=config.fft_min_hz, max_hz=config.fft_max_hz,
                                         override_hz=F0_OVERRIDE_HZ[ch])
    rois, peaks = find_uh_rois(uh["edge_signal"], fs=fs, sigma_uh=noise[ch]["sigma_uh"],
                             sigma_duh=noise[ch]["sigma_duh"], motion_interval=motion,
                             f0_hz=spectrum["f0_hz"], config=config)
    roi_results[ch] = dict(rois=rois, peaks=peaks, spectrum=spectrum)
    candidates, diagnostics = detect_roi_candidates(ud["edge_signal"], rois, fs=fs, config=config)
    detections[ch] = dict(candidates=candidates, diagnostics=diagnostics)
    edges = localize_roi_candidates(ud["clean"], ud["edge_signal"], candidates, rois,
                                   fs=fs, noise=noise[ch], config=config)
    localized[ch] = recover_missing_extrema_pairs(ud["edge_signal"], edges, rois,
                                                  fs=fs, noise=noise[ch], config=config)
    tables[ch] = build_direct_event_table(localized[ch], rois, channel=f"Udeco{ch}",
                                          sigma_udeco=noise[ch]["sigma_udeco"], fs=fs,
                                          config=config, direction_map=direction_map)
    print(f"Uh{ch}: f0={spectrum['f0_hz']:.3f} Hz | {len(rois)} ROIs | "
          f"{len(localized[ch])} edge candidates")
print(f"Motion: {motion['start_sec']:.2f}–{motion['end_sec']:.2f} s ({motion['source']})")

## 5. 绘图辅助函数

每个子图有独立幅度轴；时间轴联动。三 y 轴的第三根轴位于外侧，避免标签相互遮挡。
状态带按自身 ROI 的真实时间绘制，公共背景按 Uh2 ROI 绘制，两者互不替代。

In [ ]:
def axis_name(letter, index):
    return letter if index == 1 else f"{letter}{index}"

def axis_key(ref):
    return ref[0] + "axis" + ref[1:]

def unit(name):
    return "dimensionless" if name.startswith("Udeco") else "a.u."

def make_panels(title, groups, *, states=(), frequency=False):
    """groups: [(panel title, [axis channel labels])]; states: zero-based rows."""
    n = len(groups)
    fig, panels = go.Figure(), []
    step, next_y = 1 / n, 1
    for row, (panel_title, names) in enumerate(groups):
        low = (n - row - 1) * step
        high = low + step * 0.73
        xref = axis_name("x", row + 1)
        refs = [axis_name("y", next_y + j) for j in range(len(names))]
        next_y += len(names)
        fig.update_layout(**{axis_key(xref): dict(
            domain=[0, .82], anchor=refs[0], matches="x" if row else None,
            showticklabels=row == n - 1, title_text=("Frequency (Hz)" if frequency else "Time (s)")
            if row == n - 1 else None, showline=True, linecolor="#444444", linewidth=1,
            ticks="outside", ticklen=4, showgrid=False, zeroline=False,
            range=list(FFT_VIEW_HZ) if frequency else list(time_limits(time_s, PLOT_START_SEC, PLOT_END_SEC)))})
        for j, (name, ref) in enumerate(zip(names, refs)):
            settings = dict(domain=[low, high], anchor=xref, title_text=f"{name} ({unit(name)})",
                            title_standoff=8, tickfont=dict(size=11, color="#222222"),
                            tickformat=".3~g", nticks=4, showgrid=False, zeroline=False,
                            showline=True, linecolor=COLORS.get(name, "#555555"), linewidth=1.3,
                            ticks="outside", ticklen=4, side="left" if j == 0 else "right")
            if j:
                settings.update(overlaying=refs[0])
            if j == 2:
                settings.update(anchor="free", position=.95)
            fig.update_layout(**{axis_key(ref): settings})
        state_ref = None
        if row in states:
            state_ref = axis_name("y", next_y)
            next_y += 1
            fig.update_layout(**{axis_key(state_ref): dict(
                domain=[high + step * .035, high + step * .105], anchor=xref,
                range=[0, 1], visible=False, fixedrange=True)})
        fig.add_annotation(x=0, y=low + step * .94, xref="paper", yref="paper",
                           text=f"<b>{string.ascii_lowercase[row]}</b>   {panel_title}",
                           showarrow=False, xanchor="left", yanchor="top", font=dict(size=13))
        panels.append(dict(x=xref, ys=refs, state=state_ref))
    fig.update_layout(template="none", title=dict(text=title, x=.075, font=dict(size=17)),
                      font=dict(family="Arial, sans-serif", size=12, color="#222222"),
                      paper_bgcolor="white", plot_bgcolor="white", height=max(390, n * 265 + 90),
                      margin=dict(l=105, r=80, t=105, b=60), dragmode="pan", hovermode="x unified",
                      legend=dict(orientation="h", x=0, y=1, yanchor="bottom", font=dict(size=11),
                                  bgcolor="rgba(255,255,255,0)", borderwidth=0))
    return fig, panels

def add_signal(fig, panel, values, name, *, axis=0, x=None, color=None, width=1.25, dash=None):
    fig.add_trace(go.Scattergl(x=time_s if x is None else x, y=values, xaxis=panel["x"],
        yaxis=panel["ys"][axis], name=name, mode="lines", showlegend=False,
        line=dict(color=color or COLORS.get(name, "#334155"), width=width, dash=dash),
        hovertemplate=f"{name}<br>%{{x:.3f}} {'Hz' if x is not None else 's'}<br>%{{y:.5g}}<extra></extra>"))

def add_rois(fig, panel, rois, *, labels=False):
    shapes, annotations = [], []
    for index, roi in enumerate(rois):
        left, right = time_s[roi["left_idx"]], time_s[roi["right_idx"]]
        shapes.append(dict(type="rect", xref=panel["x"], yref=panel["ys"][0] + " domain",
                      x0=left, x1=right, y0=0, y1=1, line_width=0,
                      fillcolor=("#DDE9EF", "#F0E9DC")[index % 2], opacity=.28, layer="below"))
        for at in (left, right):
            shapes.append(dict(type="line", xref=panel["x"], yref=panel["ys"][0] + " domain",
                          x0=at, x1=at, y0=0, y1=1, opacity=.35,
                          line=dict(color="#B3BAC0", width=.6, dash="dot"),
                          layer="below"))
        if labels:
            lo, hi = time_limits(time_s, PLOT_START_SEC, PLOT_END_SEC)
            if right - left > (hi - lo) * .045:
                annotations.append(dict(x=(left + right) / 2, y=.96, xref=panel["x"],
                                   yref=panel["ys"][0] + " domain", text=roi["roi_id"],
                                   showarrow=False, font=dict(size=10, color="#555555")))
    # Replace the arrays in one operation: update_layout merges each old shape recursively.
    fig.layout.shapes = [*fig.layout.shapes, *shapes]
    fig.layout.annotations = [*fig.layout.annotations, *annotations]

def display_mode(rows):
    value = rows["roi_mode"].iloc[0] if not rows.empty and "roi_mode" in rows else "cancelled"
    return "slide" if value == "cancelled" else value

def add_states(fig, panel, ch):
    for roi in roi_results[ch]["rois"]:
        rows = tables[ch].loc[tables[ch].roi_id == roi["roi_id"]]
        mode = display_mode(rows)
        left, right = time_s[roi["left_idx"]], time_s[roi["right_idx"]]
        fig.add_trace(go.Scatter(x=[left, right, right, left, left], y=[0, 0, 1, 1, 0],
            xaxis=panel["x"], yaxis=panel["state"], mode="lines", fill="toself",
            fillcolor=STATE_COLORS[mode], line=dict(width=.6, color="white"),
            name=mode, showlegend=False, hoveron="fills",
            text=f"Udeco{ch} · {roi['roi_id']} · {mode}<br>{left:.3f}–{right:.3f} s",
            hoverinfo="text"))
        lo, hi = time_limits(time_s, PLOT_START_SEC, PLOT_END_SEC)
        if right - left >= (hi - lo) * .085:
            fig.add_annotation(x=(left + right) / 2, y=.5, xref=panel["x"], yref=panel["state"],
                               text=mode, showarrow=False, font=dict(size=10))

def add_state_key(fig):
    for mode, color in STATE_COLORS.items():
        fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", name=mode,
                                 marker=dict(symbol="square", size=10, color=color), showlegend=True))

def selected_edges(ch):
    table = tables[ch]
    if table.empty:
        return table
    selected = table.resolved & (table.is_main_rising | table.is_main_falling)
    if "overlap_duplicate" in table:
        selected &= ~table.overlap_duplicate.fillna(False).astype(bool)
    return table.loc[selected]

def add_edges(fig, panel, ch):
    values = processed[f"Udeco{ch}"]["edge_signal"]
    rows = selected_edges(ch)
    for direction, label in ((1, "Rising edge"), (-1, "Falling edge")):
        subset = rows.loc[rows.direction == direction]
        if subset.empty:
            continue
        xs, ys = [], []
        for _, item in subset.iterrows():
            left, right = int(item.start_idx), int(item.end_idx)
            xs.extend([*time_s[left:right + 1], None])
            ys.extend([*values[left:right + 1], None])
        fig.add_trace(go.Scattergl(x=xs, y=ys, xaxis=panel["x"], yaxis=panel["ys"][0],
            mode="lines", name=label, line=dict(color=EDGE_COLORS[direction], width=3),
            hovertemplate=label + "<br>%{x:.3f} s<extra></extra>"))
    if not rows.empty:
        centers = rows.center_idx.astype(int).to_numpy()
        texts = [f"{r.roi_id}: {r.event_label if pd.notna(r.get('event_label')) else 'slide'}"
                 for _, r in rows.iterrows()]
        fig.add_trace(go.Scatter(x=time_s[centers], y=values[centers], xaxis=panel["x"],
            yaxis=panel["ys"][0], mode="markers", showlegend=False, text=texts,
            marker=dict(size=6, color=[EDGE_COLORS[int(d)] for d in rows.direction],
                        symbol=["triangle-up" if d > 0 else "triangle-down" for d in rows.direction]),
            hovertemplate="%{text}<br>%{x:.3f} s<extra></extra>"))

def amplitude_spectrum(values, sampling_rate):
    values = np.asarray(values, dtype=float)
    if len(values) < 3 or not np.isfinite(values).all() or sampling_rate <= 0:
        raise ValueError("幅频谱需要至少 3 个有限采样点和正采样率。")
    window = np.hanning(len(values))
    amplitude = np.abs(np.fft.rfft((values - values.mean()) * window)) / window.sum()
    amplitude[1:] *= 2
    if len(values) % 2 == 0:
        amplitude[-1] /= 2
    return np.fft.rfftfreq(len(values), 1 / sampling_rate), amplitude

## 图 1 · 代码流程示意图（静态）

In [ ]:
flow = go.Figure()
steps = [("Read & repair", "CSV · shared sample clock"),
         ("Compute Udeco", "Calibration · paired channels"),
         ("Preprocess & baseline", "4-point filters · LOWESS"),
         ("Motion & Uh frequency", "ACC interval · Hann FFT"),
         ("Uh peaks & ROI", "Independent ROI per channel"),
         ("Udeco edges & states", "Locate · recover · classify")]
for i, (heading, detail) in enumerate(steps):
    col, row = i % 3, i // 3
    x, y = col * 3.2, 2.5 - row * 2.5
    flow.add_shape(type="rect", x0=x, x1=x + 2.7, y0=y, y1=y + 1.3,
                   fillcolor="#E9F0F4" if row == 0 else "#F3EAE0", line=dict(color="#AEBBC4", width=1))
    flow.add_annotation(x=x + 1.35, y=y + .65,
                        text=f"<b>{i + 1}. {heading}</b><br>{detail}", showarrow=False)
    if col < 2:
        flow.add_annotation(x=x + 3.13, y=y + .65, ax=x + 2.73, ay=y + .65,
                            xref="x", yref="y", axref="x", ayref="y", showarrow=True,
                            arrowhead=2, arrowwidth=1.3, text="")
# Connect the first row's last step to the second row's first step through the gutter.
flow.add_shape(type="path", path="M 7.75,2.5 L 7.75,1.95 L 1.35,1.95 L 1.35,1.55",
               line=dict(color="#555555", width=1.3))
flow.add_annotation(x=1.35, y=1.32, ax=1.35, ay=1.6, xref="x", yref="y", axref="x", ayref="y",
                    text="", showarrow=True, arrowhead=2, arrowwidth=1.3)
flow.update_layout(template="none", height=370, margin=dict(l=15, r=15, t=35, b=15),
                   font=dict(family="Arial", size=13, color="#222222"),
                   paper_bgcolor="white", plot_bgcolor="white",
                   xaxis=dict(visible=False, range=[-.2, 9.3]), yaxis=dict(visible=False, range=[-.2, 4.05]))
show_figure("01_flow", flow, static=True)

## 图 2 · Udeco 原始信号、基线与去基线信号
去基线结果为 **4 点中值滤波后的信号 − LOWESS 基线**，不是直接从原始信号相减。

In [ ]:
def plot_preprocessing(ch):
    name = f"Udeco{ch}"
    groups = [(f"{name} · raw", [name]), ("LOWESS baseline", [name]), ("Baseline removed", [name])]
    fig, panels = make_panels(f"{name} | preprocessing", groups)
    for panel, key in zip(panels, ("raw", "Bcontext", "edge_signal")):
        add_signal(fig, panel, processed[name][key], name,
                   color="#6B7280" if key == "Bcontext" else COLORS[name])
    return fig

show_figure("02_udeco2", plot_preprocessing(2))

In [ ]:
show_figure("02_udeco3", plot_preprocessing(3))

## 图 3 · 双路 Uh 幅频谱（双 y 轴）
使用运动段内去基线 Uh，幅值经 Hann 窗增益校正。主频沿用检测结果；手动覆盖主频时会明确标记。

In [ ]:
def plot_spectrum():
    fig, panels = make_panels("Uh | motion amplitude spectrum", [("Hann-window amplitude", ["Uh2", "Uh3"])],
                              frequency=True)
    panel = panels[0]
    for j, ch in enumerate((2, 3)):
        signal = processed[f"Uh{ch}"]["edge_signal"][motion["start_idx"]:motion["end_idx"]]
        freq, amplitude = amplitude_spectrum(signal, fs)
        add_signal(fig, panel, amplitude, f"Uh{ch}", axis=j, x=freq)
        result = roi_results[ch]["spectrum"]
        f0 = result["f0_hz"]
        fig.add_shape(type="line", xref="x", yref="y domain", x0=f0, x1=f0, y0=0, y1=1,
                      line=dict(color=COLORS[f"Uh{ch}"], width=1, dash="dash"))
        fig.add_annotation(x=f0, y=.96 - j * .13, xref="x", yref="y domain",
                           text=f"Uh{ch}: {f0:.3f} Hz ({result['source']})", showarrow=False,
                           xanchor="left", bgcolor="rgba(255,255,255,.85)", font=dict(size=11))
    return fig

show_figure("03_spectrum", plot_spectrum())

## 图 4 · 去基线 Uh 检峰与 ROI
每路显示自身 ROI；主峰用实心菱形，次级候选峰用浅色小圆点。

In [ ]:
def plot_peaks(ch):
    name = f"Uh{ch}"
    fig, panels = make_panels(f"{name} | peaks and ROI", [("Baseline removed", [name])])
    panel, result = panels[0], roi_results[ch]
    values = processed[name]["edge_signal"]
    add_signal(fig, panel, values, name)
    add_rois(fig, panel, result["rois"], labels=True)
    for main, label, symbol, color, size in (
        (False, "Secondary peak", "circle", "#9DA5AF", 4),
        (True, "Main peak", "diamond", COLORS[name], 7)):
        peaks = [p for p in result["peaks"] if p["is_main"] == main]
        indices = [p["peak_idx"] for p in peaks]
        if indices:
            fig.add_trace(go.Scatter(x=time_s[indices], y=values[indices], mode="markers", name=label,
                xaxis=panel["x"], yaxis=panel["ys"][0],
                marker=dict(symbol=symbol, color=color, size=size),
                text=[p["roi_id"] or "Outside ROI" for p in peaks],
                hovertemplate="%{text}<br>%{x:.3f} s<extra></extra>"))
    return fig

show_figure("04_uh2", plot_peaks(2))

In [ ]:
show_figure("04_uh3", plot_peaks(3))

## 图 5 · Udeco ROI、边沿与状态
顶部状态带使用本通道自身 ROI；悬停查看状态、ROI 和时间。仅绘制最终选用且定位成功的边沿，并去除跨 ROI 重复边沿。

In [ ]:
def plot_states(ch):
    name = f"Udeco{ch}"
    fig, panels = make_panels(f"{name} | ROI, edges and states", [("Baseline removed · own-channel ROI", [name])],
                              states=(0,))
    panel = panels[0]
    add_signal(fig, panel, processed[name]["edge_signal"], name)
    add_rois(fig, panel, roi_results[ch]["rois"])
    add_edges(fig, panel, ch)
    add_states(fig, panel, ch)
    add_state_key(fig)
    return fig

show_figure("05_udeco2", plot_states(2))

In [ ]:
show_figure("05_udeco3", plot_states(3))

## 图 6 · 多模态总览（含 ACC，五个子图）

PPG/ACC：4 点滑动平均；Uh/Udeco：去基线信号。所有子图背景均为 **Uh2 ROI**。
Udeco 状态带依照各自 ROI 的实际时间标注；因此 Udeco3 状态带边界可能与背景不同。

In [ ]:
def plot_combined(*, separate_ppg=False):
    groups = ([(name.replace("_", " "), [name]) for name in PPG_NAMES] if separate_ppg
              else [("PPG · 4-point moving average", PPG_NAMES)])
    groups += [("Uh · baseline removed", ["Uh2", "Uh3"]),
               ("Udeco2 · baseline removed", ["Udeco2"]),
               ("Udeco3 · baseline removed", ["Udeco3"])]
    state_rows = (len(groups) - 2, len(groups) - 1)
    if not separate_ppg:
        groups.append(("ACC · 4-point moving average", ACC_NAMES))
    fig, panels = make_panels("Multimodal signals | Uh2 ROI reference", groups, states=state_rows)
    if not separate_ppg:
        fig.update_layout(height=1240)  # 图 6 每行由 265 px 缩至约 230 px
    for panel, (_, names) in zip(panels, groups):
        for j, name in enumerate(names):
            values = smoothed[name] if name in smoothed else processed[name]["edge_signal"]
            add_signal(fig, panel, values, name, axis=j)
        add_rois(fig, panel, roi_results[2]["rois"])
        if names[0].startswith("Udeco"):
            add_states(fig, panel, int(names[0][-1]))
    add_state_key(fig)
    return fig

show_figure("06_multimodal", plot_combined())

## 图 7 · 分色 PPG 总览（六个子图）
三色 PPG 分别独立成图，其余处理与图 6 一致。

In [ ]:
show_figure("07_separate_ppg", plot_combined(separate_ppg=True))

## 图 8 · PPG / Uh / Udeco 基线（三个子图，无 ACC）

PPG 先做 4 点滑动平均；Uh/Udeco 先做 4 点中值滤波。三组均使用相同 LOWESS
窗口与稳健迭代次数；公共背景为 Uh2 ROI。

In [ ]:
def plot_baselines():
    groups = [("PPG baseline", PPG_NAMES), ("Uh baseline", ["Uh2", "Uh3"]),
              ("Udeco baseline", ["Udeco2", "Udeco3"])]
    fig, panels = make_panels("LOWESS baselines | Uh2 ROI reference", groups)
    for panel, (_, names) in zip(panels, groups):
        for j, name in enumerate(names):
            add_signal(fig, panel, baselines[name], name, axis=j, width=1.6)
        add_rois(fig, panel, roi_results[2]["rois"])
    return fig

show_figure("08_baselines", plot_baselines())

### 使用说明

- 修改 `PLOT_START_SEC` / `PLOT_END_SEC` 后，重新运行相应绘图单元格即可；绘图函数在调用时读取显示范围。
- 图中信号保留全时域采样点，缩放不会改变分析结果；可拖动或双击恢复初始视图。
- `FIGURES` 保存本次生成的图对象。图 2、4、5 分两路，图 1–8 共生成 11 张图；另有图 9、10 的 SVG 说明图，共 13 张。
- 无 ROI 或有效边沿时仍显示信号；不会为了填满图而补造状态。
- 如自动运动段检测失败，按报错提示同时填写 `MOTION_START_SEC` 与 `MOTION_END_SEC`。

## 图 9 · 算法流程、物理解释与佩戴应用

上层展示 Uh 与 Udeco 双分支识别，下层将信号分工、物理对应和腕表应用联系起来。
这是可编辑中文 SVG 示意图，不依赖样本数据，也不改变图 1–8。
先运行同目录的 `udeco_flowchart_assets.ipynb` 生成素材，再运行下面两个单元格即可独立显示。

In [ ]:
def plot_flowchart_cn(svg_path):
    """显示图 9；独立于所有数据分析单元格。"""
    from pathlib import Path
    from xml.etree import ElementTree as ET
    from IPython.display import SVG, display

    path = Path(svg_path).expanduser()
    if not path.is_file():
        raise FileNotFoundError(
            f"图 9 素材不存在：{path}\n请先运行同目录的 udeco_flowchart_assets.ipynb；"
            "若移动了素材，请同步修改 FLOW9_SVG_PATH。")
    content = path.read_text(encoding="utf-8")
    root = ET.fromstring(content)
    if root.tag.rsplit("}", 1)[-1] != "svg":
        raise ValueError(f"图 9 文件不是 SVG：{path}")
    figure = SVG(data=content)
    display(figure)
    return figure

In [ ]:
# 从仓库根目录或 figures/ 启动均可；只导入本工作树的模块。
from pathlib import Path
import os
import sys

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "udeco_uh_roi_edge_detection.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("请从仓库根目录或 figures/ 目录启动 Jupyter。")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

FLOW9_SVG_PATH = PROJECT_ROOT / "figures" / "assets" / "udeco_flowchart_cn.svg"
flow9 = plot_flowchart_cn(FLOW9_SVG_PATH)

### 图 9 文字与样式修改

- 打开 `udeco_flowchart_assets.ipynb` 的“只改这里”单元格。
- **流程文字**：修改 `FLOW_TEXT`；**意义说明**：直接修改 `MEANING_CARDS` 中的 `title` 和 `lines`。
- **增删卡片**：增删 `MEANING_CARDS` 列表项；每行最多三张，自动换行、等高并扩展画布。
- **样式**：修改 `FLOW_STYLE` 中的颜色、字号、宽度和间距。
- **生效步骤**：修改配置文字 → 运行素材 notebook → 重新运行上方图 9 调用单元格。
- **路径**：修改素材的 `OUTPUT_DIR` 后，同步修改这里的 `FLOW9_SVG_PATH`。
- 直接修改 SVG 会被下次生成覆盖。图 9 对象保存在 `flow9`，与 Plotly 的 `FIGURES` 分开。

`slide` 是现有 `cancelled` 的展示别名；图中的物理解释与应用方向不代表已完成或验证完整的腕表模式分类器。

## 图 10 · 稳健 LOWESS 基线估计说明（21:6 SVG）

同一多尖峰合成波形，对比常规低通与稳健 LOWESS 的基线估计。
局部拟合与残差降权可以减小孤立尖峰对基线的牵引；这个优势有适用条件，不代表所有波形上都优于低通。
在 `udeco_flowchart_assets.ipynb` 的图 10 部分可直接修改中文说明与示意参数。下方两个单元格可独立运行。

In [ ]:
def plot_lowess_explanation(svg_path):
    """独立显示 21:6 LOWESS 说明 SVG，不依赖前面的分析单元格。"""
    from pathlib import Path
    from xml.etree import ElementTree as ET
    from IPython.display import SVG, display

    path = Path(svg_path).expanduser()
    if not path.is_file():
        raise FileNotFoundError(
            f"图 10 素材不存在：{path}\n请先运行 udeco_flowchart_assets.ipynb 的图 10 部分，"
            "或修改 LOWESS_SVG_PATH。")
    content = path.read_text(encoding="utf-8")
    root = ET.fromstring(content)
    if root.tag.rsplit("}", 1)[-1] != "svg":
        raise ValueError(f"图 10 文件不是 SVG：{path}")
    figure = SVG(data=content)
    display(figure)
    return figure

In [ ]:
# 从仓库根目录或 figures/ 启动均可；只导入本工作树的模块。
from pathlib import Path
import os
import sys

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "udeco_uh_roi_edge_detection.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("请从仓库根目录或 figures/ 目录启动 Jupyter。")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

LOWESS_SVG_PATH = PROJECT_ROOT / "figures" / "assets" / "lowess_baseline_explanation.svg"
flow10 = plot_lowess_explanation(LOWESS_SVG_PATH)

### 图 10 修改与理解

**编辑素材 notebook 的 `LOWESS_TEXT` → 运行图 10 素材生成单元格 → 重新运行本图。**
`LOWESS_STYLE` 修改字号与配色，`LOWESS_DEMO` 修改合成波形及滤波参数。
默认采用 10 s LOWESS 窗口、1 次稳健迭代，与项目默认基线方法一致；低通为 4 阶、0.15 Hz、零相位。

这是基线估计示意，非等效带宽性能比较。稳健降权适用于尖峰在局部窗内占比较低的情形；密集尖峰、宽峰与真实台阶需要另行验证。
理论参考：[statsmodels LOWESS 文档](https://www.statsmodels.org/stable/generated/statsmodels.nonparametric.smoothers_lowess.lowess.html)。